# Train and test I.A for model select

### Verify if the image is corrupted

In [ ]:
from utils.verifyImages import VerifyImages

VerifyImages(path='../imagens').verifyTrainAndTest()

### Import and create itens for train

In [ ]:
import os 
from time import sleep
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import classification_report
from tqdm import tqdm
from utils.ImageDataset import ImageDataset
from utils.loadDataset import TrainDatasetImplemetation
from utils.metricsIaLearnig import Metrics
from utils.models import ImagemDetectionModels

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")   

print("Dispositivo utilizado: ", device)

In [ ]:
data_set_path = "../imagens"    

image_size = (224, 224)
batch_size = 32
data_set = TrainDatasetImplemetation(data_set_path, image_size, batch_size)
train_data = data_set.load_train_data()
test_data = data_set.load_test_data()


In [ ]:
# Define binary classficiation!
num_classes = 2
# Create the model implementation
model = ImagemDetectionModels.resnet50(num_classes)
# Pass model to the device
model = model.to(device)
# Train model 
model.train()

In [ ]:
# Define the learnin rate
learning_rate = 1e-3
# Number of epochs
num_epochs = 100
'''
    Define the loss function for multi-class classification.
    CrossEntropyLoss combines LogSoftmax and Negative Log-Likelihood Loss (NLLLoss) in one single class.
    It expects raw, unnormalized logits as input and automatically applies softmax.
    The target should contain class indices (e.g., 0, 1, 2...) corresponding to the correct class.

    Internally, for each prediction:
    1. Applies softmax to convert logits into probabilities.
    2. Takes the log of the probability for the correct class.
    3. Applies the negative log to calculate the loss.
    Intuition:
    - If the model gives high probability to the correct class → low loss (good).
    - If the model gives low probability to the correct class → high loss (bad).
    Used for classification problems with two or more classes.
'''
criterion = nn.CrossEntropyLoss()
'''
    Define the optimizer to update the model's parameters during training.
    Adam (Adaptive Moment Estimation) is an optimization algorithm that combines
    the benefits of AdaGrad and RMSProp. It adapts the learning rate for each parameter
    using estimates of the first and second moments of the gradients.

    Arguments:
    - model.parameters(): passes all trainable parameters of the model to the optimizer.
    - lr=learning_rate: sets the initial learning rate for updating the weights.

    Adam is widely used because it typically converges faster and requires less tuning
    of the learning rate compared to traditional stochastic gradient descent (SGD).
'''
optimizer = optim.Adam(model.parameters(), lr=learning_rate)


In [ ]:
metrics = Metrics(model=model, device=device, testDate=test_data)

for epoch in range(1, num_epochs + 1):
    # Create a tqdm progress bar for visualizing training progress per batch
    pbar_batch = tqdm(train_data, unit="batch")
    losses = []
    # Iterate over each batch of data
    for data in pbar_batch:
        # Set the description of the progress bar to show current epoch

        pbar_batch.set_description(f"Epoch {epoch}")
        # Unpacking and sending to GPU
        images, labels = data
        images, labels = images.to(device), labels.to(device)
        # raw scores before softmax
        scores = model(images)
        # We calculate the loss using criterion (CrossEntropyLoss).
        loss = criterion(scores, labels)
        losses.append(loss.item())
        cost = sum(losses)/len(losses)
        # Backward Pass and Weight Update
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        # Updates the progress bar
        pbar_batch.set_postfix(loss=cost)
        sleep(0.1)

    if epoch % 1 == 0:
        metrics.trainTestData()
        print(metrics)
        metrics.showAll()
        sleep(0.1)


print("model training complete")

In [ ]:
'''
    saves the trained model weights to disk, creating the directory 
    if it does not already exist.
'''
model_save_path = "weights/resnet"
if not os.path.exists(model_save_path):
    os.makedirs(model_save_path)

torch.save(model.state_dict(), model_save_path + "/resnet_model_checkpoint.pth")